# EVASPA Evapotranspiration processing

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from matplotlib.colors import ListedColormap

In [ ]:
from evaspa import ef, filter, io, merging, seb
from evaspa.config import ParamsConfig

### Common

In [ ]:
def plot_rgb(data: xr.Dataset, bands: list[str] | None = None):
    if bands is None:
        bands = ["red", "green", "blue"]
    rgb = xr.DataArray(
        np.dstack(
            (data[bands[0]].data, data[bands[1]].data, data[bands[2]].data)
        ),
        data.coords.assign(band=["r", "g", "b"]),
        dict(data.sizes, band=3),
    )
    rgb.plot.imshow(  # type: ignore
        rgb="band",
        vmin=rgb.quantile(0.01),
        vmax=rgb.quantile(0.99),
    )
    plt.title("RGB")


def plot_band(data: xr.Dataset, band: str, percentile=0):
    data[band].plot.imshow(  # type: ignore
        vmin=data[band].quantile(percentile),
        vmax=data[band].quantile(1 - percentile),
    )


def plot_mask(data: xr.Dataset, mask: str, invert=False):
    cmap = [(0.0, 0.0, 0.0, 0.0)]
    if len(np.unique(data[mask].data)) > 1:
        cmap = [(1.0, 0.0, 0.0, 1.0), (0.0, 0.0, 0.0, 0.0)]
    if invert:
        cmap = cmap[::-1]
    valid_cmap = ListedColormap(cmap)
    data[mask].plot(cmap=valid_cmap, add_colorbar=False)


def plot(
    data: xr.Dataset,
    var_name: str | None = None,
    model: ef.EFModel | None = None,
    save: bool = False,
):
    """
    Plot
    """
    # dimensions
    # gridspec inside gridspec
    fig = plt.figure(constrained_layout=True, figsize=(4, 3))

    ax = plt.gca()

    if var_name is None and model is None:
        msg = "Provide either var_name or model"
        raise ValueError(msg)
    lst = data["lst"].data

    if var_name is not None:
        var = data[var_name].data
        # plot all points
        ax.scatter(var, lst, s=20, alpha=1, color="moccasin", clip_on=False)

    # plot edges
    elif model is not None:
        print("toto")
        var_name = model.var
        var = data[model.var].data
        # plot all points
        ax.scatter(var, lst, s=20, alpha=1, color="moccasin", clip_on=False)
        # edges coordinates
        var_max = np.round(np.ceil(np.nanmax(var) * 10) / 10, decimals=1)
        var_min = np.round(np.floor(np.nanmin(var) * 10) / 10, decimals=1)

        var_arr = np.linspace(var_min, var_max, num=20)
        tw_arr = model.twet(var_arr)
        td_arr = model.tdry(var_arr)

        ax.plot(
            var_arr,
            td_arr,
            color="tab:red",
            linewidth=3,
            label="dry edge",
            zorder=1,
        )
        ax.plot(
            var_arr,
            tw_arr,
            color="tab:blue",
            linewidth=3,
            label="wet edge",
            zorder=1,
        )

    # Labels
    ax.set_xlabel(var_name)
    ax.set_ylabel("Land Surface Temperature K")

    # title
    ax.set_title("Evaporative fraction method", fontsize=12)

    # savefig
    if save:
        destfile = "ef.png"
        fig.savefig(destfile)

    plt.show()

### Input

Data

In [ ]:
data = io.read_data_from_file("../tests/data/modis_test.tif")

In [ ]:
data

In [ ]:
plot_band(data, band="lst")

Configuration

In [ ]:
params = {
    "filtering": {},
    "ef": {
        "check": {"threshold": 0.02},
        "models": [
            {
                "name": "EF1",
                "dry_edge": {
                    "type": "LinearEdge",
                    "config": {
                        "interval_type": "density",
                        "interval_nb": 20,
                        "percentile": [95, 100],
                        "selection": "median",
                    },
                },
                "wet_edge": {
                    "type": "LinearEdge",
                    "config": {
                        "interval_type": "density",
                        "interval_nb": 20,
                        "percentile": [0, 5],
                        "selection": "median",
                    },
                },
                "var": "albedo",
            },
            {
                "name": "EF8",
                "dry_edge": {
                    "type": "LinearEdge",
                    "config": {
                        "interval_type": "density",
                        "interval_nb": 20,
                        "percentile": [95, 100],
                        "selection": "median",
                    },
                },
                "wet_edge": {
                    "type": "FlatEdge",
                    "config": {"selection": "min"},
                },
                "var": "albedo",
            },
        ],
        "options": {"selection": False, "merging": "mean"},
    },
    "seb": {"use_topo": False, "models": ["kustas"]},
}

## Run step by step 

Validate parameters config

In [ ]:
# Validate parameters config
params_config = ParamsConfig.model_validate(params)

In [ ]:
params_config

In [ ]:
pprint(params_config.model_dump(), sort_dicts=False)

Filter

In [ ]:
# Filter data
data["valid"] = filter.determine_valid_pixels(
    data, **params_config.filtering.model_dump()
)

In [ ]:
if (data["valid"].data == 0).sum() == 0:
    print("No invalid pixels")
else:
    plot_mask(data=data, mask="valid", invert=True)

Check variability

In [ ]:
ef.check_variability(
    data["lst"], mask=data["valid"], **params_config.ef.check.model_dump()
)

Compute EF

In [ ]:
models, options = ef.initialize(params_config.ef.model_dump())

In [ ]:
ef_xr, ef_inst_xr = ef.run(
    models, data, mask="valid", **params_config.ef.options.model_dump()
)

In [ ]:
ef.EFModel.create(ef_xr["EF1"].attrs)

In [ ]:
plot(data, var_name="albedo")

In [ ]:
plot(data, model=ef.EFModel.create(ef_xr["EF1"].attrs))

In [ ]:
plot_band(ef_xr, band="EF1")

In [ ]:
ef.EFModel.create(ef_xr["EF8"].attrs)

In [ ]:
plot(data, model=ef.EFModel.create(ef_xr["EF8"].attrs))

In [ ]:
plot_band(ef_xr, band="EF8")

In [ ]:
plot_band(ef_inst_xr, band="ef")

Compute LE

In [ ]:
print(f"Constant Stefan Boltzmann = {seb.CST_SB}")

*Compute net radiation*

In [ ]:
rn_xr = seb.create_net_radiation(data)

In [ ]:
rn_xr

In [ ]:
print(f"Max = {np.nanmax(rn_xr.rn)}")
print(f"Min = {np.nanmin(rn_xr.rn)}")

In [ ]:
plot_band(rn_xr, band="rn")

*Compute G/Rn ratio*

In [ ]:
ratio_xr = seb.create_ratio(data, models=params_config.seb.models)

In [ ]:
ratio_xr

In [ ]:
print(f"Max = {np.nanmax(ratio_xr.kustas)}")
print(f"Min = {np.nanmin(ratio_xr.kustas)}")

In [ ]:
plot_band(ratio_xr, band="kustas")

In [ ]:
le_xr = seb.create_le(ef_xr, rn_xr, ratio_xr)

In [ ]:
le_xr

In [ ]:
plot_band(le_xr, band="EF1_rn_kustas")

In [ ]:
plot_band(le_xr, band="EF8_rn_kustas")

Merge LE

In [ ]:
le_inst = merging.merge_to_dataset(le_xr, method=merging.MergeMethod.MEAN)

In [ ]:
le_inst

In [ ]:
plot_band(le_inst, band="merged")